<a href="https://colab.research.google.com/github/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-/blob/main/notebooks/velocity_breakdown.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Velocity breakdown: is the signal spatial or temporal?

Implied velocity is distance divided by time gap. In Sparkov, merchant locations are random points near the cardholder's home, while fraud arrives in bursts of 7–16 transactions on one day. So velocity may be strong only because the time gaps inside a burst are short, not because of geography.

We test this in two parts, using the same protocol as `time_honest_evaluation.ipynb` (time-based validation, Random Forest Tuned, five seeds, 95% day-block bootstrap intervals):

1. **Breakdown.** We train the model on distance only, time gap only and velocity only, and on the six features with velocity replaced by time gap.
2. **Location-shuffle test.** Within each card, we shuffle the merchant locations between that card's transactions, keeping all times unchanged. We then recompute velocity and retrain. If the scores barely change, location adds nothing to velocity.

## 1. Environment setup
The same setup as our earlier notebooks. We type credentials in at runtime, so they are never saved in the notebook.

In [1]:
# 1.1 Download the Sparkov dataset from Kaggle
import os
from getpass import getpass

os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ')
os.environ['KAGGLE_KEY'] = getpass('Kaggle API key: ')

!kaggle datasets download -d kartik2112/fraud-detection -p /content/data/sparkov --unzip

Kaggle username: Subbareddy_subbz
Kaggle API key: ··········
Dataset URL: https://www.kaggle.com/datasets/kartik2112/fraud-detection
License(s): CC0-1.0
100% 202M/202M [00:01<00:00, 182MB/s]



In [2]:
# 1.2 Clone the project repository
github_token = getpass('Paste your GitHub token: ')
repo_url = f'https://{github_token}@github.com/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-.git'
!git clone {repo_url}

Paste your GitHub token: ··········
Cloning into 'Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-'...
remote: Enumerating objects: 176, done.
remote: Counting objects: 100% (176/176), done.
remote: Compressing objects: 100% (168/168), done.
remote: Total 176 (delta 109), reused 17 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (176/176), 11.54 MiB | 12.84 MiB/s, done.
Resolving deltas: 100% (109/109), done.


In [3]:
# 1.3 Move into the repository and place the data in data/
%cd Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-
!mv /content/data/sparkov data/

/content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-


### 1.4 Saving progress to Google Drive
We save each finished model and seed to Google Drive straight away, so if Colab disconnects, re-running the notebook skips everything already finished.

In [4]:
# 1.4 Connect Google Drive for saving progress
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/fraud_checkpoints/velocity_breakdown'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('checkpoints ->', CHECKPOINT_DIR)

Mounted at /content/drive
checkpoints -> /content/drive/MyDrive/fraud_checkpoints/velocity_breakdown


In [5]:
# 1.5 Imports and project helpers
import sys
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

sys.path.append(os.getcwd())
from src.split import split_sparkov
from src.utils import fit_category_risk, apply_category_risk, add_prev_txn_features
from src.evaluation import (chronological_val_split, run_seeds, bootstrap_model,
                            summarise, paired_difference, format_ci)

## 2. Features
`add_prev_txn_features` compares each transaction with the same card's previous one and gives us three columns: `dist_from_prev_txn_km`, `time_gap_hours` and `implied_velocity_kmh`. We log-transform all three, as we did for velocity before.

A card's first transaction has no previous one. As in our earlier notebooks, we set its velocity to 0, and we do the same for distance. For the time gap, 0 would wrongly look like an instant repeat purchase, so we use -1 instead, which the trees can keep separate from real gaps.

We combine train and test into one timeline and give it a fresh index, because the train and test indexes overlap. A `_split` column records where each row came from.

In [6]:
# 2.1 One timeline for train + test, with a fresh index
sparkov_train, sparkov_test = split_sparkov()
sparkov_train['_split'] = 'train'
sparkov_test['_split']  = 'test'
full_raw = (pd.concat([sparkov_train, sparkov_test])
            .sort_values('trans_date_trans_time')
            .reset_index(drop=True))
TIME = 'trans_date_trans_time'
print(full_raw.shape)

Train: 1,296,675 rows, from 2019-01-01 00:00:18 to 2020-06-21 12:13:37
Test:  555,719 rows, from 2020-06-21 12:14:25 to 2020-12-31 23:59:34
(1852394, 24)


In [7]:
# 2.2 Movement features (distance, time gap, velocity) and the other features
def add_movement_features(df):
    df = add_prev_txn_features(df, 'cc_num', TIME, 'merch_lat', 'merch_long')
    df['dist_log']     = np.log1p(df['dist_from_prev_txn_km'].fillna(0))
    df['gap_log']      = np.log1p(df['time_gap_hours']).fillna(-1)   # -1 = no previous transaction
    df['velocity_log'] = np.log1p(df['implied_velocity_kmh'].fillna(0))
    return df

def add_other_features(df):
    ts = pd.to_datetime(df[TIME])
    df['is_night']    = ts.dt.hour.isin([22, 23, 0, 1, 2, 3]).astype(int)
    df['amt_log']     = np.log1p(df['amt'])
    df['age']         = (ts - pd.to_datetime(df['dob'])).dt.days / 365.25
    df['gender_flag'] = (df['gender'] == 'M').astype(int)
    df['day']         = ts.dt.floor('D')   # used only by the bootstrap
    return df

full = add_other_features(add_movement_features(full_raw))

### 2.3 Time-based split and `category_risk`
The same split as `time_honest_evaluation.ipynb`: the last 20% of the training period is validation, and `category_risk` is fitted on the training part only. The split depends only on time, so every run in this notebook uses exactly the same rows.

In [8]:
# 2.3 Split by time, then fit category_risk on the training part only
def split_and_encode(full_df):
    train = full_df[full_df['_split'] == 'train']
    test  = full_df[full_df['_split'] == 'test'].copy()
    train_sub, val, cutoff = chronological_val_split(train, TIME, val_frac=0.2)
    risk_map = fit_category_risk(train_sub, 'category', 'is_fraud')
    for df in (train_sub, val, test):
        df['category_risk'] = apply_category_risk(df, 'category', risk_map)
    return train_sub, val, test, cutoff

train_sub, val, test, cutoff = split_and_encode(full)
y_tr, y_val, y_test = train_sub['is_fraud'], val['is_fraud'], test['is_fraud']
test_days = test['day'].values

print('cutoff:', cutoff)
for name, df in [('train_sub', train_sub), ('val', val), ('test', test)]:
    print(f"{name:9s} rows={len(df):>9,}  fraud rate={df['is_fraud'].mean():.4%}")

cutoff: 2020-03-06 00:10:57.200000
train_sub rows=1,036,901  fraud rate=0.5756%
val       rows=  259,774  fraud rate=0.5921%
test      rows=  555,719  fraud rate=0.3860%


### 2.4 A first look: how do fraud and genuine transactions differ?
Median distance, time gap and velocity for fraud and genuine transactions in the training part. If fraud differs mainly in time gap and hardly at all in distance, that already points to a temporal signal.

In [9]:
# 2.4 Medians by class (training part only)
look = train_sub[train_sub['time_gap_hours'].notna()]
look.groupby('is_fraud')[['dist_from_prev_txn_km', 'time_gap_hours', 'implied_velocity_kmh']].median().round(2)

,dist_from_prev_txn_km,time_gap_hours,implied_velocity_kmh
is_fraud,,,
0,100.70,4.57,20.11
1,98.24,1.33,64.23


## 3. Feature sets
| Run | Features | What it shows |
|---|---|---|
| Baseline | the six features | our reference result |
| Distance only | `dist_log` | how much location carries on its own |
| Time gap only | `gap_log` | how much timing carries on its own |
| Velocity only | `velocity_log` | the combined feature on its own |
| Six, gap instead of velocity | velocity replaced by `gap_log` | whether time gap can fully replace velocity |
| Six, no velocity | the five other features | how much velocity adds at all |

All runs use the Random Forest Tuned settings. Each split normally considers 3 features (`max_features=3`), but a single-feature model only has 1, so for those runs each split uses that one feature.

In [10]:
# 3.1 Feature sets and the model
SIX = ['amt_log', 'category_risk', 'is_night', 'velocity_log', 'age', 'gender_flag']
FEATURE_SETS = {
    'Baseline (six features)':     SIX,
    'Distance only':               ['dist_log'],
    'Time gap only':               ['gap_log'],
    'Velocity only':               ['velocity_log'],
    'Six, gap instead of velocity': ['amt_log', 'category_risk', 'is_night', 'gap_log', 'age', 'gender_flag'],
    'Six, no velocity':            ['amt_log', 'category_risk', 'is_night', 'age', 'gender_flag'],
}

def make_rf(n_features):
    # Random Forest Tuned, with max_features capped at the number of features
    return lambda s: RandomForestClassifier(n_estimators=200, max_features=min(3, n_features),
                                            min_samples_leaf=5, random_state=s, n_jobs=-1)

SEEDS = [42, 43, 44, 45, 46]
RUNS_TO_DO = list(FEATURE_SETS)   # the order they are trained in

## 4. Breakdown runs
Each run takes about 4–7 minutes per seed (single-feature runs are faster), so this section takes about 2.5 hours in total.

In [11]:
# 4.1 Train every feature set with every seed (finished runs are loaded from Drive)
runs = {}
for name in RUNS_TO_DO:
    cols = FEATURE_SETS[name]
    runs[name] = run_seeds(name, make_rf(len(cols)),
                           train_sub[cols], y_tr, val[cols], y_val, test[cols], SEEDS,
                           checkpoint_dir=CHECKPOINT_DIR)

Baseline (six features) seed 42: loaded from checkpoint
Baseline (six features) seed 43: loaded from checkpoint
Baseline (six features) seed 44: loaded from checkpoint
Baseline (six features) seed 45: loaded from checkpoint
Baseline (six features) seed 46: loaded from checkpoint
Distance only seed 42: loaded from checkpoint
Distance only seed 43: loaded from checkpoint
Distance only seed 44: loaded from checkpoint
Distance only seed 45: loaded from checkpoint
Distance only seed 46: loaded from checkpoint
Time gap only seed 42: loaded from checkpoint
Time gap only seed 43: loaded from checkpoint
Time gap only seed 44: loaded from checkpoint
Time gap only seed 45: loaded from checkpoint
Time gap only seed 46: loaded from checkpoint
Velocity only seed 42: loaded from checkpoint
Velocity only seed 43: loaded from checkpoint
Velocity only seed 44: loaded from checkpoint
Velocity only seed 45: loaded from checkpoint
Velocity only seed 46: loaded from checkpoint
Six, gap instead of velocity s

## 5. Location-shuffle test
Within each card, we randomly reorder the merchant locations (latitude and longitude together) across that card's transactions. Every location is still a real location for that card, and all times, amounts and labels stay unchanged. Only the link between *where* and *when* is broken.

We then recompute distance and velocity and retrain the six-feature model. Each seed uses its own shuffle (seed 42 → shuffle 42, and so on), so the seed SD also covers the randomness of the shuffle.

In [12]:
# 5.1 Shuffle merchant locations within each card
def shuffle_locations_within_card(df, seed):
    df = df.copy()
    rng = np.random.default_rng(seed)
    by_card = df.sort_values('cc_num', kind='stable')        # rows grouped by card
    shuffled = (by_card.assign(_r=rng.random(len(df)))
                       .sort_values(['cc_num', '_r']))        # same cards, random order inside each
    df.loc[by_card.index, ['merch_lat', 'merch_long']] = shuffled[['merch_lat', 'merch_long']].values
    return df

# check: each card keeps exactly the same set of locations
check = shuffle_locations_within_card(full_raw, 42)
before = full_raw.groupby('cc_num')['merch_lat'].apply(lambda s: tuple(sorted(s)))
after  = check.groupby('cc_num')['merch_lat'].apply(lambda s: tuple(sorted(s)))
assert (before == after).all()
print('share of rows whose location moved:', round((check['merch_lat'] != full_raw['merch_lat']).mean(), 4))
del check, before, after

share of rows whose location moved: 0.9995


In [13]:
# 5.2 Retrain the six-feature model on shuffled locations, one shuffle per seed
SHUFFLE_NAME = 'Six, shuffled locations'
runs[SHUFFLE_NAME] = []
shuffle_medians = []
for s in SEEDS:
    shuffled_full = add_other_features(add_movement_features(shuffle_locations_within_card(full_raw, s)))
    tr_s, val_s, test_s, _ = split_and_encode(shuffled_full)
    assert len(test_s) == len(test) and (test_s['is_fraud'].values == y_test.values).all()

    look_s = tr_s[tr_s['time_gap_hours'].notna()]
    shuffle_medians.append(look_s.groupby('is_fraud')['implied_velocity_kmh'].median().rename(s))

    runs[SHUFFLE_NAME] += run_seeds(SHUFFLE_NAME, make_rf(len(SIX)),
                                    tr_s[SIX], tr_s['is_fraud'], val_s[SIX], val_s['is_fraud'],
                                    test_s[SIX], [s], checkpoint_dir=CHECKPOINT_DIR)

Six, shuffled locations seed 42: loaded from checkpoint
Six, shuffled locations seed 43: loaded from checkpoint
Six, shuffled locations seed 44: loaded from checkpoint
Six, shuffled locations seed 45: loaded from checkpoint
Six, shuffled locations seed 46: loaded from checkpoint


The test rows come out in the same order for the shuffled and real data (checked by the assertion above), so all runs can be scored on the same bootstrap resamples.

**Median velocity (km/h) by class, real vs shuffled locations (training part):**

In [14]:
# 5.3 Does the fraud-vs-genuine velocity gap survive the shuffle?
real = look.groupby('is_fraud')['implied_velocity_kmh'].median().rename('real')
pd.concat([real] + shuffle_medians, axis=1).round(1)

,real,42,43,44,45,46
is_fraud,,,,,,
0,20.1,20.1,20.1,20.2,20.1,20.2
1,64.2,64.0,61.3,63.4,63.2,64.2


## 6. Results with confidence intervals
### 6.1 Results table
As in `time_honest_evaluation.ipynb`: 500 day-block resamples, the same `BOOT_SEED` for every run, seed average with a 95% interval.

In [15]:
# 6.1 Bootstrap each run and build the results table
N_BOOT = 500
BOOT_SEED = 2026

boots, summaries = {}, {}
for name, seed_runs in runs.items():
    pairs = [(r['test_proba'], r['threshold']) for r in seed_runs]
    boots[name] = bootstrap_model(y_test, pairs, test_days, n_boot=N_BOOT, boot_seed=BOOT_SEED)
    summaries[name] = summarise(y_test, pairs, boots[name])
    summaries[name]['Val PR-AUC'] = np.mean([r['val_pr_auc'] for r in seed_runs])
    print(f"{name}: done")

summary_df = pd.DataFrame(summaries).T
table = pd.DataFrame({
    'Val PR-AUC': summary_df['Val PR-AUC'].astype(float).round(4),
    'PR-AUC [95% CI]': [format_ci(r, 'PR-AUC') for _, r in summary_df.iterrows()],
    'PR-AUC seed SD': summary_df['PR-AUC seed SD'].astype(float).round(4),
    'F2 [95% CI]':  [format_ci(r, 'F2')  for _, r in summary_df.iterrows()],
    'MCC [95% CI]': [format_ci(r, 'MCC') for _, r in summary_df.iterrows()],
    'Thresholds (per seed)': summary_df['Thresholds'],
})
table

Baseline (six features): done
Distance only: done
Time gap only: done
Velocity only: done
Six, gap instead of velocity: done
Six, no velocity: done
Six, shuffled locations: done


,Val PR-AUC,PR-AUC [95% CI],PR-AUC seed SD,F2 [95% CI],MCC [95% CI],Thresholds (per seed)
Baseline (six features),0.8769,"0.8341 [0.8140, 0.8529]",0.0003,"0.7892 [0.7689, 0.8056]","0.7180 [0.6941, 0.7385]","0.15, 0.16, 0.15, 0.19, 0.14"
Distance only,0.0060,"0.0039 [0.0035, 0.0044]",0.0000,"0.0167 [0.0134, 0.0198]","0.0011 [-0.0018, 0.0039]","0.01, 0.03, 0.01, 0.04, 0.04"
Time gap only,0.0078,"0.0047 [0.0041, 0.0054]",0.0000,"0.0330 [0.0278, 0.0385]","0.0238 [0.0190, 0.0285]","0.01, 0.01, 0.01, 0.01, 0.01"
Velocity only,0.0069,"0.0043 [0.0038, 0.0050]",0.0000,"0.0247 [0.0210, 0.0286]","0.0097 [0.0062, 0.0128]","0.01, 0.02, 0.01, 0.01, 0.01"
"Six, gap instead of velocity",0.8837,"0.8444 [0.8250, 0.8626]",0.0006,"0.7943 [0.7760, 0.8113]","0.7241 [0.7009, 0.7438]","0.17, 0.18, 0.17, 0.16, 0.15"
"Six, no velocity",0.8625,"0.8167 [0.7955, 0.8362]",0.0005,"0.7747 [0.7558, 0.7928]","0.7196 [0.6993, 0.7381]","0.22, 0.19, 0.16, 0.21, 0.21"
"Six, shuffled locations",0.8758,"0.8333 [0.8138, 0.8515]",0.0009,"0.7830 [0.7622, 0.8004]","0.7094 [0.6844, 0.7287]","0.19, 0.13, 0.15, 0.17, 0.14"


### 6.2 Change from the baseline
For each run, we calculate its score minus the baseline's score on every resample. If the 95% interval includes zero, we cannot tell the run apart from the baseline.

In [16]:
# 6.2 Paired differences against the baseline (run minus baseline)
BASE = 'Baseline (six features)'
diff_rows = []
for name in boots:
    if name == BASE:
        continue
    for metric in ['PR-AUC', 'F2', 'MCC']:
        d = paired_difference(boots[name], boots[BASE], metric)
        diff_rows.append({'Run': name, 'Metric': metric, **{k: round(v, 4) for k, v in d.items()}})
diff_df = pd.DataFrame(diff_rows).rename(columns={'P(A > B)': 'P(run > baseline)'})
diff_df

,Run,Metric,mean diff,CI low,CI high,P(run > baseline)
0,Distance only,PR-AUC,-0.8297,-0.8489,-0.8104,0.000
1,Distance only,F2,-0.7719,-0.7891,-0.7531,0.000
2,Distance only,MCC,-0.7164,-0.7377,-0.6928,0.000
3,Time gap only,PR-AUC,-0.8289,-0.8481,-0.8097,0.000
4,Time gap only,F2,-0.7555,-0.7714,-0.7387,0.000
5,Time gap only,MCC,-0.6936,-0.7136,-0.6706,0.000
6,Velocity only,PR-AUC,-0.8292,-0.8486,-0.8101,0.000
7,Velocity only,F2,-0.7638,-0.7801,-0.7464,0.000
8,Velocity only,MCC,-0.7077,-0.7289,-0.6843,0.000
9,"Six, gap instead of velocity",PR-AUC,0.0103,0.0057,0.0146,1.000


In [17]:
# 6.3 Save the results tables
os.makedirs('results', exist_ok=True)
summary_df.to_csv('results/velocity_breakdown_summary.csv')
table.to_csv('results/velocity_breakdown_table.csv')
diff_df.to_csv('results/velocity_breakdown_paired_differences.csv', index=False)
print('saved to results/')

saved to results/


## 7. Discussion
*(We complete this section after the run.)*

We look at four points:
- whether time gap alone comes close to velocity alone, while distance alone is weak;
- whether replacing velocity with time gap in the six-feature model makes little or no difference;
- whether shuffling the locations leaves the scores unchanged (paired interval including zero);
- how much velocity adds compared with the model without it.

**Limitations:**
- Sparkov is synthetic, and we designed these tests around how its generator creates locations and fraud bursts. Whether the same holds on real data is tested separately on IEEE-CIS.
- The shuffle keeps each card's set of locations, so it removes the link between location and time, but not each card's general area.